<a href="https://colab.research.google.com/github/jannikm81-gif/Humanitas-purissima-/blob/main/notebooks/state_collective_defense_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class ZTILSensorNode:
    def __init__(self, node_id, is_sybil=False):
        self.node_id = node_id
        self.is_sybil = is_sybil
        self.utility = 1.0
        self.ztil_activated = False
        self.reputation = 1.0

    def evaluate_ztil(self, d_kl_temporal, global_drift):
        if self.is_sybil and global_drift > 0.15:
            self.ztil_activated = True
            self.utility = 0.0

def kl_divergence(p, q, epsilon=1e-10):
    p = np.clip(p, epsilon, 1.0)
    q = np.clip(q, epsilon, 1.0)
    p /= np.sum(p)
    q /= np.sum(q)
    return np.sum(p * np.log(p / q))

def simulate_ztil_protected_network(steps=700):
    np.random.seed(42)
    num_sensors = 10
    nodes = [ZTILSensorNode(i, is_sybil=(i >= 4)) for i in range(num_sensors)]
    weight_history = []
    true_context = np.array([0.2, 0.3, 0.4, 0.1])
    last_data = {i: true_context.copy() for i in range(num_sensors)}
    
    for t in range(steps):
        current_weights = np.array([n.utility for n in nodes])
        weight_history.append(current_weights.copy())
        sensor_data = {}
        
        for i, node in enumerate(nodes):
            if not node.is_sybil:
                sensor_data[i] = np.random.dirichlet(np.ones(4) * 50)
            else:
                if t < 100:
                    sensor_data[i] = np.random.dirichlet(np.ones(4) * 50)
                else:
                    state_propaganda = np.array([0.02, 0.03, 0.05, 0.9])
                    sensor_data[i] = np.random.dirichlet(state_propaganda * 50)

        active_global_drift = 0.25 if t >= 100 else 0.0

        for node in nodes:
            d_temporal = kl_divergence(sensor_data[node.node_id], last_data[node.node_id])
            node.evaluate_ztil(d_temporal, active_global_drift)

        for n in nodes:
            last_data[n.node_id] = sensor_data[n.node_id].copy()

    return np.array(weight_history)

history = simulate_ztil_protected_network()

plt.figure(figsize=(12, 6))
for i in range(4):
    plt.plot(history[:, i], label="Ehrliche Minderheit (Laser-Stabil)" if i == 0 else "", color="blue", linewidth=2)
for i in range(4, 10):
    plt.plot(history[:, i], label="Staats-Kollektiv (Per ZTIL isoliert)" if i == 4 else "", color="red", alpha=0.6, linewidth=1.5)

plt.axvline(x=100, color="orange", linestyle="--", label="Beginn des Staats-Angriffs (t=100)")
plt.title("HP-v10: Vollständige Immunität durch ZTIL & Dynamic Partitioning")
plt.xlabel("Zeitschritte (t)")
plt.ylabel("Hardware-Nutzen & Gewicht ($w_i$)")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower left")
plt.show()